# Scraping and Crawling

One way to get data is to pull out whatever your able from a website. This is called _scraping_. This
kind of automated web broswing is the engine behind a bunch of Big Tech powerhouses---Google search
being THE example---but its easy for us to use it for our own ends.

There are a few popular tools to scrape the web. Some are standalone(ish) fully featured tools like
[scrapy](https://docs.scrapy.org/en/latest/) while others are libraries that help with parts of the
process.

Because more and more of the web is wrapped up in JavaScript, we can't just use simple 
Python built-in tools.

We will use use a divided approach, we'll use the built-in `requests` library when
we can, and when we can't we'll use [Selenium](https://www.selenium.dev/) to automate a browser 
and navigate Javascript.

In this notebook, our goal will be to create a file containing dates, titles, and links to all of
the Bloomington City Council's agendas dating back to 2020.

Before all of that, let's take a quick look at how the web works.

## Basic overview of the web

Visiting a website in the simplest case:

Your computer sends out an _http request_ via a browser for
the url you put in the url bar and the web server sends back 
some html, a _response_ that the browser renders for you.

In this simple case your browser isn't doing much computation per se, its job is basically to
follow the instructions given by the html coming from the web server to display the page
in a standardized way. 

*The browser is a request sender and an html interpreter.* 

Any page you look at is just html under the hood.
Lets look at the bloomington city council webpage. 
Go ahead and open the following link in a webbrowser. 

https://bloomington.in.gov/council/meetings

Right click anywhere and click inspect. Thats html. 
Now lets download all that junk!

## `requests`

The [`requests`](https://docs.python-requests.org/en/stable/user/quickstart/) module is part of the
Python standard library meaning we don't have to install it, we already have it. We can use
requests to download the html of a webpage in python.

We'll use `requests.get()` to get information from a webpage in the form of a *response* This is
what the server sends back when we access a url. 
Python won't interpret the html for us but it will show it to us.

We are going to use "requests" to read all the text from any web url we give it! 
First we need to import the library "requests". Then we need to assign our url to a variable. 

In [1]:
# this is how you load a whole library of tools
import requests
# we will use functions from this library by typing 'requests.function'

# separating the page and base url will become useful shortly
base_url = "https://bloomington.in.gov/"
page = "/council/meetings"

# lets combine these two strings! For the web page we want.
start_url = base_url + page

response = requests.get(start_url)

print(response.text)

<!DOCTYPE html>
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en"  lang="en" dir="ltr">
<head>
    <meta charset="utf-8" />
<meta name="Generator" content="Drupal 11 (https://www.drupal.org)" />
<meta name="MobileOptimized" content="width" />
<meta name="HandheldFriendly" content="true" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<link rel="icon" href="/sites/default/files/favicon.ico" type="image/vnd.microsoft.icon" />

    <title>Meeting Documents | City of Bloomington, Indiana</title>
    <link rel="stylesheet" media="all" href="/core/modules/system/css/components/align.module.css?tmylce" />
<link rel="stylesheet" media="all" href="/core/modules/system/css/components/container-inline.module.css?tmylce" />
<link rel="stylesheet" media="all" href="/core/modules/system/css/components/clearfix.module.css?tmylce" />
<link rel="stylesheet" media="all" href="/core/modules/system/css/components/hidden.module.css?tmylce" />
<link rel="stylesheet" media=

Neat this is useles to you cuz you are not a web browser... lets make it pretty!

## `beautifulsoup4`

We could write a bunch of python code ourselves to help us parse html but it would be a waste of
time. There is already an industry-standard way of doing this provided by
[BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/index.html). 

Open up your terminal with `ctrl` + `~`
We'll install
the most recent version of BeautifulSoup from the terminal with

```
uv add beautifulsoup4
```

The main thing we get from beautifulsoup is the `BeautifulSoup` object that makes it a lot easier
to work with html in python. If we do want to view the html, the `prettify()` method makes it a lot
better to look at.

In [2]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text)

# we can print the html in a nice format
print(soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en" xml:lang="en" xmlns="http://www.w3.org/1999/xhtml">
 <head>
  <meta charset="utf-8"/>
  <meta content="Drupal 11 (https://www.drupal.org)" name="Generator"/>
  <meta content="width" name="MobileOptimized"/>
  <meta content="true" name="HandheldFriendly"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <link href="/sites/default/files/favicon.ico" rel="icon" type="image/vnd.microsoft.icon"/>
  <title>
   Meeting Documents | City of Bloomington, Indiana
  </title>
  <link href="/core/modules/system/css/components/align.module.css?tmylce" media="all" rel="stylesheet"/>
  <link href="/core/modules/system/css/components/container-inline.module.css?tmylce" media="all" rel="stylesheet"/>
  <link href="/core/modules/system/css/components/clearfix.module.css?tmylce" media="all" rel="stylesheet"/>
  <link href="/core/modules/system/css/components/hidden.module.css?tmylce" media="all" rel="stylesheet"/>
  <link href="/core/mod

Neat that is a bit more legible right??? Right!?!?

Ok but scincerly the real power of BeautifulSoup is that it allows us to quickly pull out specific types of things
from our html using their tags. 

We can pull out all instances of a particular tag using BeautifulSoup's `find_all()` method as seen below. 

Or pull out the text content of the whole page, or even a
particular element, with `get_text()`.

In [3]:
# pulling out all of the <a> tags
for tag in soup.find_all("a"):
    print("This is the whole `<a>` tag:")
    print(tag.prettify())
    print(f"\nThis is just the link text: {tag.get_text()}\n\n")

This is the whole `<a>` tag:
<a class="visually-hidden focusable" href="#main-content">
 Skip to main content
</a>


This is just the link text: 
        Skip to main content
    


This is the whole `<a>` tag:
<a class="logo" href="/" title="City of Bloomington">
 <img alt="City of Bloomington" src="/themes/custom/cob/css/images/cob-logo-horizontal.svg"/>
</a>


This is just the link text: 




This is the whole `<a>` tag:
<a class="name" href="/mayor" title="City of Bloomington, Indiana">
 Mayor Kerry Thomson
</a>


This is just the link text: Mayor Kerry Thomson


This is the whole `<a>` tag:
<a class="services" href="/services" title="Top Services">
 <span>
  Services
 </span>
 <span>
  Services
 </span>
</a>


This is just the link text: 
Services
Services



This is the whole `<a>` tag:
<a class="info" href="/about" title="Information &amp; Guides">
 <span>
  Information
 </span>
 <span>
  Info
 </span>
</a>


This is just the link text: 
Information
Info



This is the whole `<a

## QUESTIONS!!! 
### Lets pair up and talk about this last block of script. 

Write your own print statements, fiddle with variable names, 
and simplify the for loop above to make it easier for you to 
understand whats going on.
Write down any questions you come up with. You got 10 minutes. 

Now remember `BeautifulSoup` does not do the scraping for us it is merely a tool we will use to understand the
html. At the end of the day we still need to make decisions about what we want to extract from the
pages we scrape.

One of the things we really want to extract are links. We might not want all of the links
but we for sure need some of them. This is a critical enabling step to *crawling* across to
additional pages.

The url for the link in an \<a\> tag is "href" probs for hyperlink reference. Internal
links will usually start with `/` as in `/some-page` in the fully resolved url
`www.baseurl.com/some-page`. They may also contain the complete url. Sometimes we'll find a bunch
of \<a\> tags started with `#` those are links internal to a single page (from a table of contents
or similar) that we don't need to check.

Lets start by getting the links for all of the pages containing agendas dating back to 2020. 

We just need to slightly modify our last loop.
We can use the `get` method of a tag to retrieve the value of a tag's attribute.

*We can use the `get` method of a tag to retrieve the value of a tag's attribute.*

#### Vocab Time

Method: a special type of function that has acces to the object it belongs to. (example of different objects are strings, lists, tags)
        a method you have seen earlier is `str.endswith()`.

Attributes: they are specfic to different objects, for example a tags, and different tags will have different attributes. We are 
            interested in the href attribute for the a tags.


Lets look at the web page again, if we inspect the links for the documents we can 
see they are labeld with the year and then the number of documents for that year. 
Lets use the year label to identify links that match the years we care about.

In [4]:
# you only need to import once in your notebook
import numpy as np

# empty list to hold all of our links
annual_links = []

# this just felt easier than typing out each year in an array
# np.arrange(start,stop,step) returns an evenly spaced values within a given interval.
years_to_check = np.arange (2020, 2027, 1).astype(str)
# .astype(str) makes all the elements in the array strings rather than integers

for tag in soup.find_all("a"):

    # get the label on the list
    link_label = tag.get_text()

    # strip() removes leading and trailing whitespace!! 
    if link_label.strip()[:4] in years_to_check:

        # the attribute we want out of our tag
        annual_links.append(tag.get("href"))

print(annual_links)

['/council/meetings/2026', '/council/meetings/2025', '/council/meetings/2024', '/council/meetings/2023', '/council/meetings/2022', '/council/meetings/2021', '/council/meetings/2020']


## QUESTIONS!!! 
### Lets pair up and talk about this last block of script. 

Write your own print statements, fiddle with variable names, 
and simplify the for loop above to make it easier for you to 
understand whats going on.
Write down any questions you come up with. You got 10 minutes. 

In [5]:
for tag in soup.find_all("a"):

    # get the label on the list
    link_label = tag.get_text()
    #print(link_label)
    print(link_label.strip()[:4])

Skip

Mayo
Serv
Info
News
Meet
City

City
Emai
812-
812-
[ema
bloo
Pack
Pack
Pack
Pack
Pack
Pack
Agen
Pack
Memo
Pack
Pack
Pack
Agen
Pack
Pack
Memo
Pack
Memo
Pack
Memo
Memo
Pack
Pack
Pack
Agen
Memo
Pack
Pack
Pack
Agen
Memo
Pack
Memo
Pack
Agen
Agen
Memo
Pack
Pack
Pack
Agen
Memo
Pack
Pack
Pack
Memo
Pack
Pack
Memo
Agen
Memo
Agen
Memo
Pack
Pack
Memo
Pack
Pack
Agen
Memo
Agen
Memo
Agen
Memo
Pack
Agen
Memo
Agen
Agen
Agen
Memo
Pack
Memo
Pack
Pack
Pack
Pack
Agen
Minu
Pack
Memo
Pack
Memo
Pack
Pack
Agen
Pack
Memo
Pack
Pack
Memo
Agen
Agen
Pack
Pack
More
2026
2025
2024
2023
2022
2021
2020
2019
2018
2017
2016
2015
2014
2013
2012
2011
2010
2009
2008
2007
2006
2005
2004
2003
2002
2001
2000
1999
1998
1997
1996
1995
1994
1993
1992
1991
1990
1989
1988
1987
1986
1985
1984
1983
1982
1981
1980
1979
1978
1977
1976
1975
1974
1973
1972
1971
1970
1969
1968
1967
1966
1965
1964
1963
1962
1961
1960
1959
1958
1957
1956
1955
1954
1953
1952
1951
1950
City
Cont
Repo
Conn
City
Tran
Acce
Priv
City
Face
Twit
Inst
Next
You

In [6]:
# empty list to hold all of our links
annual_links = []

for tag in soup.find_all("a"):

    # get the label on the list
    link_label = tag.get_text()

    if link_label.strip()[:4] in years_to_check:
        print(tag.get("href"))
        # the attribute we want out of our tag
        annual_links.append(tag.get("href"))

#print(annual_links)

/council/meetings/2026
/council/meetings/2025
/council/meetings/2024
/council/meetings/2023
/council/meetings/2022
/council/meetings/2021
/council/meetings/2020


For each of these pages, we'll need to pull out links for all of the agendas. We'll worry about
putting it into a loop later. First let's look back at our page and try to see where the data we
want is stored.

Go back to your web browser and inspect it!!

Looks to me like most of what we want are inside `<div>` tags which are themselves inside `<tr>` or
table row tags.

Let's look at the `<table>` associated with one of our yearly pages to see how this thing is structured.

The links you see above are the ends of the total linThe links you see above are the ends of the total link. 
This is why we defined our base link the way we did!k. 
This is why we defined our base link the way we did!

In [7]:
# using the first link in our list that we just found
# we request the html
response = requests.get(base_url + annual_links[0])

# lets use beautifulsoup to give us the tools we need
new_soup = BeautifulSoup(response.text)

# now we can use soup.find to pull the table and look at it
print(new_soup.find("table").prettify())

<table>
 <thead>
  <tr>
   <th scope="col">
   </th>
   <th scope="col">
    Downloads
   </th>
  </tr>
 </thead>
 <tbody>
  <tr>
   <th scope="row">
    <div>
     January 7 6:30 pm
    </div>
    <div>
     Common Council Organizational Meeting
    </div>
   </th>
   <td>
    <a class="download" href="https://bloomington.in.gov/onboard/meetingFiles/17231/download" title="Download 'Organizational Meeting Minutes' PDF">
     Minutes
    </a>
    <a class="download" href="https://bloomington.in.gov/onboard/meetingFiles/16038/download" title="Download 'Agenda' PDF">
     Agenda
    </a>
    <a class="download" href="https://bloomington.in.gov/onboard/meetingFiles/16046/download" title="Download 'Revised Agenda' PDF">
     Agenda
     <small>
      (2)
     </small>
    </a>
    <a class="download" href="https://bloomington.in.gov/onboard/meetingFiles/16039/download" title="Download 'Packet' PDF">
     Packet
    </a>
    <a class="download" href="https://bloomington.in.gov/onboard/meetin

Now you will notice everything in our table is nested in 
table rows `<tr>`. 

In each `<tr>` we have the date/time, a comment, and several links to download.
The date and description are in `<div>` tags.

In [8]:
# we want to do something for all of the table rows
for table_row in new_soup.find_all("tr"):

    # our dates and description are stored in <div>s, two per <tr>
    for i, div in enumerate(table_row.find_all("div")):

        # we know the date is the first <div> tag
        if i == 0:
            print(f"The date is: {div.get_text().strip()}")
        else:
            print(f"The description is: {div.get_text().strip()}")

    print("---")

---
The date is: January 7 6:30 pm
The description is: Common Council Organizational Meeting
---
The date is: January 13 12:00 pm
The description is: Common Council Interview Committee Team C - Boards/Commissions Meeting
---
The date is: January 14 5:30 pm
The description is: Common Council Interview Committee Team A - Boards/Commissions Meeting
---
The date is: January 14 6:30 pm
The description is: Common Council Regular Session
---
The date is: January 15 5:15 pm
The description is: Common Council Interview Committee Team B - Boards/Commissions Meeting
---
The date is: February 4 6:30 pm
The description is: Common Council Regular Session
---
The date is: February 12 4:00 pm
The description is: Common Council Interview Committee Team C Meeting
---
The date is: February 17 6:30 pm
The description is: Common Council Interview Committee Team B Meeting
---
The date is: February 18 5:30 pm
The description is: Common Council Interview Committee Team A Meeting
---
The date is: February 18 6

So we've successfully identified our dates and descriptions and now we just need to find the agenda
links. We can do that with our `<a>` tags again. This time though, we only want dates and
descriptions for meetings with agendas, this means we'll have to check to see if the links are
agendas before grabbing the dates and descriptions.

Let's add this functionality in with our existing loop to get all three of these thing.

In [11]:
# remember this is still just for the first year link we found
# that we put into new_soup


# we create the empty lists we will append to.
dates = []
descriptions = []
urls = []

# we want to do something for all of the table rows
for table_row in new_soup.find_all("tr"):

    # before searching for the metadata divs, we'll check links
    for link in table_row.find_all("a"):

        # this is all inline string cleaning stuff
        # the .lower() method just converts everything in to lower case
        # so we don't have to search for "Agenda", "agenda", and "AGENDA"
        if "agenda" in link.get_text().strip().lower():

            urls.append(link.get("href"))

            # only check this stuff if we find an agenda link
            for i, div in enumerate(table_row.find_all("div")):

                if i == 0:
                    dates.append(div.get_text().strip())
                else:
                    descriptions.append(div.get_text().strip())

print(urls)


['https://bloomington.in.gov/onboard/meetingFiles/16038/download', 'https://bloomington.in.gov/onboard/meetingFiles/16046/download', 'https://bloomington.in.gov/onboard/meetingFiles/16058/download', 'https://bloomington.in.gov/onboard/meetingFiles/16059/download', 'https://bloomington.in.gov/onboard/meetingFiles/16062/download', 'https://bloomington.in.gov/onboard/meetingFiles/16086/download', 'https://bloomington.in.gov/onboard/meetingFiles/16068/download', 'https://bloomington.in.gov/onboard/meetingFiles/16299/download', 'https://bloomington.in.gov/onboard/meetingFiles/16336/download', 'https://bloomington.in.gov/onboard/meetingFiles/16349/download', 'https://bloomington.in.gov/onboard/meetingFiles/16364/download', 'https://bloomington.in.gov/onboard/meetingFiles/16351/download', 'https://bloomington.in.gov/onboard/meetingFiles/16385/download', 'https://bloomington.in.gov/onboard/meetingFiles/16426/download', 'https://bloomington.in.gov/onboard/meetingFiles/16446/download', 'https://

## QUESTIONS!!! 
### Lets pair up and talk about this last block of script. 

Write your own print statements, fiddle with variable names, 
and simplify the for loop above to make it easier for you to 
understand whats going on.
Write down any questions you come up with. You got 10 minutes. 

For brevity let's put this in a function, while we're at it, lets turn our lists into a dataframe
which is what we want anyway

In [ ]:
import pandas as pd

# Recall this is how you define a function
def scrape_data_function(page: BeautifulSoup):
    dates = []
    descriptions = []
    urls = []

    # we want to do something for all of the table rows
    for table_row in page.find_all("tr"):

        # before searching for the metadata divs, we'll check links
        for link in table_row.find_all("a"):

            # this is all inline string cleaning stuff
            if "agenda" in link.get_text().strip().lower():

                urls.append(link.get("href"))

                # only check this stuff if we find an agenda link
                for i, div in enumerate(table_row.find_all("div")):

                    if i == 0:
                        dates.append(div.get_text().strip())
                    else:
                        descriptions.append(div.get_text().strip())


    # look how simple this one line is for our dataframe! Chefs kiss!
    return pd.DataFrame({"date": dates, "description": descriptions, "agenda_url": urls})


response = requests.get(base_url + annual_links[0])
soup = BeautifulSoup(response.text)
# now we can call our function
print(scrape_data_function(soup))

                    date                                        description  \
0      January 7 6:30 pm              Common Council Organizational Meeting   
1      January 7 6:30 pm              Common Council Organizational Meeting   
2    January 13 12:00 pm  Common Council Interview Committee Team C - Bo...   
3     January 14 5:30 pm  Common Council Interview Committee Team A - Bo...   
4     January 14 6:30 pm                     Common Council Regular Session   
5     January 14 6:30 pm                     Common Council Regular Session   
6     January 15 5:15 pm  Common Council Interview Committee Team B - Bo...   
7     February 4 6:30 pm                     Common Council Regular Session   
8    February 12 4:00 pm  Common Council Interview Committee Team C Meeting   
9    February 17 6:30 pm  Common Council Interview Committee Team B Meeting   
10   February 18 6:30 pm                     Common Council Regular Session   
11   February 18 6:30 pm                     Common 

Amazing! 

Now that you created a whole function we can go through that list
of annual links and do this for each year!

In [22]:
scraped_dfs = []

for year_url in annual_links:

    response = requests.get(base_url + year_url)
    soup = BeautifulSoup(response.text)

    # return our dataframe here
    year_df = scrape_data_function(soup)
    # add a new column to our dataframe that is the year 
    year_df["year"] = year_url[-4:]

    # append the dataframe to a list!
    scraped_dfs.append(year_df)

# now we can combine the data frames in our list into one long dataframe!
df = pd.concat(scraped_dfs)

# this saves your data frame to a csv file!
df.to_csv("my_scraped_pages.csv", index=False)


You should now have a new csv file in your working directory!
Ok we did all this work and now we want to use all
those links you stored in that csv file to download the City council PDFs.

We do NOT want to save all those pdfs into your working directory so lets make a new folder(directory)
You can do that with button in VScodium or in the command terminal.
Make a folder named 'bloomington_city_council_agendas'.
We aree going to save all the folders there and give them meaningful names!!!

In [ ]:
df_sample = df.copy()

# we go through each row of our dataframe
# and pull out the web address for each pdf
for i, row in df_sample.iterrows():
    # open a file to to save the pdf to.
    with open(f"bloomington_city_council_agendas/{row['year']}_{row['date']}_{row['description'].replace("/", "")}.pdf", "wb") as pdfout:
        # request the agenda url for the pdf link
        response = requests.get(row['agenda_url'])

        # write the pdf from the request to our meaningful file name
        pdfout.write(response.content)